# Treinar a voz da Nora no Piper (Kaggle)

Ajuste fino de uma voz do Piper com o conjunto gerado por `nora voz gerar-dataset`, na GPU grátis do Kaggle
(30 h por semana). **Roda sozinho em segundo plano**: você aperta um botão, pode fechar o navegador e volta
horas depois para baixar a voz. Guia completo: `docs/TREINAR_VOZ.md` no repositório da Nora.

**Uma vez só**

1. Crie uma conta em [kaggle.com](https://www.kaggle.com) e **confirme o celular** (Settings → Phone
   verification). Sem isso, a GPU e a internet ficam bloqueadas.

**Cada treino**

1. **Dataset:** Create → New Dataset → envie o `nora_v1.zip` (o Kaggle descompacta sozinho) → Create.
2. **Notebook:** Create → New Notebook → File → Import Notebook → escolha este arquivo.
3. No painel da direita (**Notebook → Session options** ou **Settings**):
   - **Accelerator:** GPU P100 (ou GPU T4 x2, se a P100 não aparecer);
   - **Internet:** ligada;
   - **Add Input** (ou "+ Add Data") → aba **Your Datasets** → o dataset do passo 1.
4. Confira a célula 1 (o nome da voz e o número de épocas).
5. **Save Version → Save & Run All (Commit) → Save.** Pronto: pode fechar a página.
6. Volte depois de umas horas: **View Active Events** (ou a página do notebook → aba **Output**) e baixe
   `pt_BR-norav1-medium.onnx`, `pt_BR-norav1-medium.onnx.json` e `teste.wav`.

O treino para sozinho antes do limite de 12 h do Kaggle e ainda assim exporta a voz.

**Licenças:** o Piper (`piper1-gpl`, GPL-3.0) é baixado aqui, à parte, e nunca entra no repositório da Nora.
As vozes base do Piper costumam derivar de dados só para uso não comercial: a voz treinada é para **uso pessoal**
e fica fora do Git (`models/`). Veja `docs/LICENCAS.md`.

## 1. Configuração

In [ ]:
NOME_VOZ = "nora_v1"  # mesmo nome usado no gerar-dataset
MAX_EPOCAS = 1000  # ~3 a 4 h; a voz costuma ficar boa entre 800 e 1.500
LOTE = 16
LIMITE_DE_TEMPO = "00:11:00:00"  # dias:horas:min:seg; para antes das 12 h do Kaggle e exporta
# Vazio = escolhe uma voz pt_BR "medium"; ou o caminho de um .ckpt do piper-checkpoints
VOZ_BASE = ""
FRASE_DE_TESTE = "Oi! Hoje de manhã eu organizei a sua agenda. Você quer ouvir os compromissos?"

## 2. GPU

In [ ]:
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise RuntimeError(
        "Sem GPU. No painel da direita, em Accelerator, escolha GPU P100 ou GPU T4 x2 "
        "(precisa do celular confirmado na conta do Kaggle)."
    )
print("GPU:", torch.cuda.get_device_name(0))

## 3. Instalar o Piper (uns 5 minutos)

In [ ]:
import os
import subprocess

PIPER = "/tmp/piper1-gpl"
if not os.path.isdir(PIPER):
    !git clone -q --depth 1 https://github.com/OHF-Voice/piper1-gpl.git {PIPER}
if not os.path.isdir(PIPER):
    raise RuntimeError("Não baixou o Piper: ligue a Internet no painel da direita (Settings).")
%cd {PIPER}
!pip -q install scikit-build cmake ninja cython
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace
%cd /kaggle/working

teste = "from piper import espeakbridge; from piper.train.vits.monotonic_align import maximum_path"
resultado = subprocess.run(["python3", "-c", teste], capture_output=True, text=True)
if resultado.returncode != 0:
    raise RuntimeError("A instalação do Piper falhou:\n" + resultado.stderr[-2000:])
print("Piper pronto.")

## 4. Conjunto de treino

In [ ]:
import pathlib
import zipfile

import soundfile as sf

ENTRADA = pathlib.Path("/kaggle/input")
metas = sorted(ENTRADA.rglob("metadata.csv"))
if not metas:  # se o Kaggle não descompactou o zip
    for arquivo in ENTRADA.rglob("*.zip"):
        with zipfile.ZipFile(arquivo) as arquivo_zip:
            arquivo_zip.extractall("/tmp/dataset")
    metas = sorted(pathlib.Path("/tmp/dataset").rglob("metadata.csv"))
if not metas:
    raise RuntimeError(
        "Não achei o metadata.csv: adicione o dataset em Add Input (painel da direita)."
    )
META = metas[0]
WAVS = META.parent / "wavs"
print("Conjunto:", META.parent)

boas, faltando, taxas, segundos = [], [], set(), 0.0
for linha in META.read_text(encoding="utf-8-sig").splitlines():
    if not linha.strip():
        continue
    ident, _, texto = linha.partition("|")
    wav = WAVS / (ident if ident.endswith(".wav") else f"{ident}.wav")
    if not wav.is_file() or not texto.strip():
        faltando.append(ident)
        continue
    info = sf.info(wav)
    taxas.add(info.samplerate)
    segundos += info.duration
    boas.append(f"{ident}|{texto.strip()}")

META_OK = pathlib.Path("/tmp/metadata_ok.csv")
META_OK.write_text("\n".join(boas) + "\n", encoding="utf-8")
print(f"{len(boas)} frases, {segundos / 60:.1f} min de fala.")
if faltando:
    print(f"Ignoradas (sem áudio ou sem texto): {', '.join(faltando[:20])}")
assert taxas == {22050}, f"Os áudios precisam estar em 22.050 Hz; encontrei {taxas}."

## 5. Voz base (ponto de partida do ajuste fino)

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "rhasspy/piper-checkpoints"
if VOZ_BASE.startswith("/"):
    BASE = VOZ_BASE
else:
    ckpts = [a for a in list_repo_files(REPO, repo_type="dataset") if a.endswith(".ckpt")]
    pt_br = [a for a in ckpts if "pt_BR" in a and "/medium/" in a]
    print("Vozes pt_BR medium disponíveis:", *pt_br, sep="\n  ")
    escolhida = VOZ_BASE or (
        pt_br[0] if pt_br else next(a for a in ckpts if "en_US/lessac/medium" in a)
    )
    print("Usando:", escolhida)
    BASE = hf_hub_download(REPO, escolhida, repo_type="dataset")

## 6. Treinar

Para no fim das épocas ou no `LIMITE_DE_TEMPO`, o que vier primeiro. Para **continuar** um treino anterior
do Kaggle: adicione a saída (Output) daquela versão em Add Input; o `last.ckpt` dela é encontrado sozinho.

In [ ]:
import shlex
import shutil

SAIDA = pathlib.Path("/kaggle/working")
TREINO = pathlib.Path("/tmp/treino")
CONFIG = TREINO / "config.json"
TREINO.mkdir(parents=True, exist_ok=True)

opcoes = {
    "--data.voice_name": NOME_VOZ,
    "--data.csv_path": META_OK,
    "--data.audio_dir": WAVS,
    "--model.sample_rate": 22050,
    "--data.espeak_voice": "pt-br",
    "--data.cache_dir": "/tmp/cache",
    "--data.config_path": CONFIG,
    "--data.batch_size": LOTE,
    "--trainer.max_epochs": MAX_EPOCAS,
    "--trainer.max_time": LIMITE_DE_TEMPO,
    "--trainer.devices": 1,
    "--trainer.default_root_dir": TREINO,
}
comando = ["python3", "-m", "piper.train", "fit"]
for opcao, valor in opcoes.items():
    comando += [opcao, str(valor)]
anteriores = sorted(ENTRADA.rglob("last.ckpt"))
if anteriores:
    print("Continuando de", anteriores[0])
    comando += ["--ckpt_path", str(anteriores[0])]
    config_anterior = anteriores[0].parent / "config.json"
else:
    print("Começando a partir da voz base.")
    comando += ["--model.warmstart_ckpt", BASE]

get_ipython().system(shlex.join(comando))  # noqa: F821 (existe no Kaggle)

ultimos = sorted(TREINO.glob("lightning_logs/version_*/checkpoints/last.ckpt"))
if not ultimos:
    raise RuntimeError("O treino não gerou checkpoint: veja o erro acima.")
CKPT = ultimos[-1]
shutil.copy(CKPT, SAIDA / "last.ckpt")  # para continuar depois, se quiser
if not CONFIG.is_file() and anteriores and config_anterior.is_file():
    shutil.copy(config_anterior, CONFIG)
shutil.copy(CONFIG, SAIDA / "config.json")
print("Treino terminado:", CKPT)

## 7. Exportar e testar

In [ ]:
import wave

from IPython.display import Audio, display
from piper import PiperVoice

NOME_ARQUIVO = f"pt_BR-{NOME_VOZ.replace('_', '')}-medium"
ONNX = SAIDA / f"{NOME_ARQUIVO}.onnx"
exportar = ["python3", "-m", "piper.train.export_onnx", "--checkpoint", CKPT, "--output-file", ONNX]
get_ipython().system(shlex.join(map(str, exportar)))  # noqa: F821 (existe no Kaggle)
shutil.copy(CONFIG, f"{ONNX}.json")

voz = PiperVoice.load(str(ONNX))
teste = SAIDA / "teste.wav"
with wave.open(str(teste), "wb") as arquivo_wav:
    voz.synthesize_wav(FRASE_DE_TESTE, arquivo_wav)
display(Audio(str(teste)))
print("Na aba Output:", ONNX.name, "e", ONNX.name + ".json")
print(f'No local.toml da Nora: voz = "{NOME_ARQUIVO}"')

## 8. Levar para a Nora

Baixe da aba **Output** os dois arquivos da voz e copie para `B:\Nora\models\piper\`. No `config\local.toml`:

```toml
[voz]
motor = "piper"

[voz.piper]
voz = "pt_BR-norav1-medium"
```

Ouça o `teste.wav` antes. Se a voz ainda soar metálica, rode de novo com a saída desta versão em
**Add Input** (o treino continua do `last.ckpt`) e um `MAX_EPOCAS` maior.